# Pengambilan kisaran harga coffee shop dari GoFood

Tujuan: memberi dasar **perkiraan revenue kasar** per kawasan (arahan dosen: "pakai toko yang sudah ada, revenue kasar berdasarkan menu yang dijual").

**Sumber & batasan etis**
- Hanya halaman daftar publik `gofood.co.id/bandung/<kecamatan>-restaurants/<kategori>` yang diizinkan `robots.txt` (yang dilarang: `/api/*`, `/_next/data/*`, halaman ulasan).
- Halaman detail outlet (menu lengkap) dilindungi captcha → **tidak diambil**; scraper berhenti otomatis kalau bertemu captcha / HTTP 403 / 429.
- Scraper juga berhenti kalau server memutus koneksi (tanda pembatasan). Jalankan ulang lain hari untuk melanjutkan: halaman yang sudah ada di cache tidak diunduh lagi.
- Jeda 3 detik per permintaan, setiap halaman disimpan (cache) sehingga hanya diambil sekali.
- Data yang disimpan: nama outlet, koordinat, tag masakan, rating, dan **tingkat harga** GoFood.

**Tingkat harga GoFood** (label resmi dari situs): 1 = di bawah Rp16rb, 2 = Rp16–40rb, 3 = Rp40–100rb, 4 = di atas Rp100rb (per menu/orang). 0 = tidak diketahui.

Keterbatasan: setiap halaman daftar hanya menampilkan 12 outlet (tanpa paginasi), jadi hasilnya **sampel**, bukan sensus.

In [1]:
import sys
print("Python kernel:", sys.executable)
%pip install -q geopandas pandas numpy requests rapidfuzz

Python kernel: /private/tmp/claude-501/-Users-teyok-Downloads-ExtractingDataSkripsi/0724e23e-948c-4914-b645-096e1971d3ec/scratchpad/venv/bin/python



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import json, re, time
from datetime import datetime, timezone
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, requests
from rapidfuzz import fuzz

ROOT = Path.cwd()
EXTRACT = ROOT.parent / "notebook_extract_cafe" / "data"
CACHE = ROOT / "data/raw/gofood"; CACHE.mkdir(parents=True, exist_ok=True)
OUT = ROOT / "data/interim/gofood_outlet.csv"
COFFEE_CSV = EXTRACT / "final/coffeeshop_bandung.csv"
KECAMATAN_FILE = EXTRACT / "raw/kecamatan_kota_bandung.geojson"
BATAS_FILE = EXTRACT / "raw/batas_kota_bandung.geojson"

UA = "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0 Safari/537.36"
JEDA_DETIK = 3
CRS_METER = 32748

# slug kecamatan GoFood yang berada di (atau berbatasan dengan) Kota Bandung
SLUG = ["andir", "arcamanik", "astana-anyar", "babakan-ciparay", "bandung-kidul", "bandung-kulon", "bandung-wetan",
        "batununggal", "bojong-loa-kaler", "bojong-loa-kidul", "cibeunying-kaler", "cibeunying-kidul", "cibiru",
        "cicadas", "cicendo", "cidadap", "coblong", "kiaracondong", "lengkong", "margacinta", "rancasari", "regol",
        "sukajadi", "sukasari", "sumur-bandung", "ujung-berung", "kodya-bandung"]
KATEGORI = ["", "coffee_shop", "minuman", "best_seller", "most_loved", "affordable_price", "near_me",
            "24_hours", "snacks_jajanan", "roti", "healthy_food"]
HARGA_LEVEL = {1: "< Rp16rb", 2: "Rp16–40rb", 3: "Rp40–100rb", 4: "> Rp100rb"}

## 1. Ambil halaman daftar (cache + jeda + berhenti kalau diblokir)

In [3]:
class Diblokir(Exception):
    pass

sesi = requests.Session(); sesi.headers["User-Agent"] = UA

def ambil(slug, kat):
    f = CACHE / f"{slug}__{kat or 'semua'}.json"
    if f.exists():
        return json.loads(f.read_text(encoding="utf-8")), False
    url = f"https://gofood.co.id/bandung/{slug}-restaurants" + (f"/{kat}" if kat else "")
    try:
        r = sesi.get(url, timeout=30)
    except (requests.ConnectionError, requests.Timeout) as e:
        # server memutus koneksi = tanda pembatasan; berhenti, jangan dicoba ulang/diakali
        raise Diblokir(f"{url} -> koneksi diputus ({type(e).__name__})")
    if r.status_code in (403, 429) or "captcha" in r.text[:3000].lower():
        raise Diblokir(f"{url} -> HTTP {r.status_code}")
    if r.status_code == 404:
        data = {"url": url, "outlets": []}
    else:
        r.raise_for_status()
        m = re.search(r'<script id="__NEXT_DATA__"[^>]*>(.*?)</script>', r.text, re.S)
        outlets = json.loads(m.group(1))["props"]["pageProps"].get("outlets") or [] if m else []
        data = {"url": url, "diambil": datetime.now(timezone.utc).isoformat(timespec="seconds"), "outlets": outlets}
    f.write_text(json.dumps(data, ensure_ascii=False), encoding="utf-8")
    return data, True

halaman, baru = [], 0
try:
    for slug in SLUG:
        for kat in KATEGORI:
            d, dari_internet = ambil(slug, kat)
            halaman.append((slug, kat, d))
            if dari_internet:
                baru += 1
                time.sleep(JEDA_DETIK)
except Diblokir as e:
    print("BERHENTI (diblokir, tidak dicoba mengakali):", e)
print(f"Halaman: {len(halaman)} ({baru} baru diunduh, sisanya dari cache)")

Halaman: 297 (0 baru diunduh, sisanya dari cache)


## 2. Ratakan menjadi tabel outlet

In [4]:
rows = []
for slug, kat, d in halaman:
    for o in d["outlets"]:
        c = o.get("core", {})
        loc = c.get("location") or {}
        tags = [t.get("displayName", "") for t in c.get("tags", [])]
        rows.append(dict(uid=c.get("uid"), nama=c.get("displayName", ""), lat=loc.get("latitude"), lon=loc.get("longitude"),
                         tag=";".join(tags), price_level=o.get("priceLevel") or 0,
                         rating=(o.get("ratings") or {}).get("average"), rating_n=(o.get("ratings") or {}).get("total"),
                         path=o.get("path", ""), slug=slug, kategori_gofood=kat or "semua",
                         diambil=d.get("diambil")))
g = pd.DataFrame(rows)
g = (g.groupby("uid", as_index=False)
       .agg(nama=("nama", "first"), lat=("lat", "first"), lon=("lon", "first"), tag=("tag", "first"),
            price_level=("price_level", "max"), rating=("rating", "first"), rating_n=("rating_n", "first"),
            path=("path", "first"), muncul_di=("kategori_gofood", lambda s: ";".join(sorted(set(s)))),
            diambil=("diambil", "max")))
KOPI = r"COFFEE|COFFE|KOFFIE|KOPI|NGOPI|ESPRESSO|ROASTERY|BARISTA|\bBREW"
g["kopi"] = g.tag.str.contains(r"\bKopi\b", case=False) | g.nama.str.upper().str.contains(KOPI)
g = gpd.GeoDataFrame(g, geometry=gpd.points_from_xy(g.lon, g.lat), crs=4326)

kota = gpd.read_file(BATAS_FILE).geometry.union_all()
kec = gpd.read_file(KECAMATAN_FILE)[["kecamatan", "geometry"]]
g = g[g.within(kota)]
g = gpd.sjoin(g, kec, how="left", predicate="within").drop(columns="index_right")
print(f"Outlet unik di Kota Bandung: {len(g)} | coffee/kopi: {int(g.kopi.sum())}")
print("Tingkat harga (outlet kopi):", g[g.kopi].price_level.map(HARGA_LEVEL).fillna("tidak diketahui").value_counts().to_dict())

Outlet unik di Kota Bandung: 1089 | coffee/kopi: 301
Tingkat harga (outlet kopi): {'Rp16–40rb': 128, 'Rp40–100rb': 85, 'tidak diketahui': 55, '< Rp16rb': 32, '> Rp100rb': 1}


## 3. Cocokkan dengan dataset coffee shop v2
Pasangan dianggap sama kalau jaraknya ≤ 150 m **dan** kemiripan nama (`token_set_ratio`) ≥ 90.

In [5]:
def norm(s):
    s = re.sub(r"\(.*?\)", " ", str(s).upper())
    s = re.sub(r",.*$", " ", s)                 # buang ", Nama Cabang"
    return re.sub(r"[^A-Z0-9 ]", " ", s).strip()

cs = pd.read_csv(COFFEE_CSV)
cs = gpd.GeoDataFrame(cs, geometry=gpd.points_from_xy(cs.lon, cs.lat), crs=4326).to_crs(CRS_METER)
gk = g[g.kopi].to_crs(CRS_METER).reset_index(drop=True)
pas = gpd.sjoin_nearest(cs[["nama", "geometry"]].reset_index(), gk[["uid", "nama", "price_level", "geometry"]],
                        max_distance=150, distance_col="jarak_m", lsuffix="cs", rsuffix="gf")
pas["skor_nama"] = [fuzz.token_set_ratio(norm(a), norm(b)) for a, b in zip(pas.nama_cs, pas.nama_gf)]
pas = pas[pas.skor_nama >= 90].sort_values(["index", "skor_nama"], ascending=[True, False]).drop_duplicates("index")
print(f"Coffee shop v2 yang ditemukan di GoFood: {len(pas)} dari {len(cs)} ({100*len(pas)/len(cs):.0f}%)")
display(pas[["nama_cs", "nama_gf", "jarak_m", "skor_nama", "price_level"]].head(15))

Coffee shop v2 yang ditemukan di GoFood: 28 dari 400 (7%)


,nama_cs,nama_gf,jarak_m,skor_nama,price_level
19,BAGI KOPI SIGNATURE DAGO,"Bagi Kopi Signature, Dago",14.735342,100.0,2
41,DE.U COFFEE MEKARWANGI,"De.U Coffee Mekarwangi, Jalan Mekarlaksana No. 8b",111.221188,100.0,0
44,DJOURNAL COFFEE,"Djournal Coffee by ISMAYA, Paris Van Java - Re...",23.375615,100.0,3
114,KOPI KENANGAN,"Kopi Kenangan, Pajajaran Bandung",4.953592,100.0,2
115,KOPI KENANGAN,"Kopi Kenangan, Graha POS RE Martadinata",72.648909,100.0,2
130,KOPIKITAKU,"Kopikitaku, Pahlawan",4.202949,100.0,3
154,MILES COFFEE,"Miles Coffee, Turangga",20.889042,100.0,3
159,MONDAY COFFEE DAGO,"Monday Coffee, Dago",29.428187,100.0,3
167,ONE EIGHTY COFFEE AND MUSIC,"One Eighty Coffee Music, Ganesha",33.350232,100.0,3
175,POINT COFFEE,"Point Coffee, Indomaret Burangrang",16.090076,100.0,2


## 4. Simpan

In [6]:
g["match_v2"] = g.uid.isin(pas.uid)
g["nama_v2"] = g.uid.map(pas.sort_values("skor_nama", ascending=False).drop_duplicates("uid").set_index("uid").nama_cs)
g["harga_label"] = g.price_level.map(HARGA_LEVEL)
g.drop(columns="geometry").to_csv(OUT, index=False)
ringkas = (g[g.kopi & (g.price_level > 0)].groupby("kecamatan").price_level
           .agg(n_outlet="size", median_level="median", mean_level="mean"))
print("Disimpan:", OUT.relative_to(ROOT))
ringkas.sort_values("n_outlet", ascending=False)

Disimpan: data/interim/gofood_outlet.csv


,n_outlet,median_level,mean_level
kecamatan,,,
Lengkong,31,2.0,2.225806
Bandung Wetan,16,3.0,2.750000
Coblong,15,2.0,2.133333
Cibeunying Kidul,13,2.0,2.384615
Sukajadi,13,2.0,2.076923
Kiaracondong,13,2.0,2.000000
Cibiru,12,2.0,2.000000
Sukasari,11,3.0,2.272727
Regol,10,2.0,2.400000
